# **Dataset Maker**

Follow this notebook to transform your data into a dataset that can be used to train the model.

## 📋 **1. Required Data**

RNeNcodec requires three types of data: **audio files**, **annotations**, and **parameter specifications**. Each one is explained below.

### **1.1 🎵 Audio Files**

Most audio file formats and sampling rates are supported, although the final model will always operate at 24kHz.

**Supported Audio Formats**: `.wav`, `.mp3`, `.flac`, `.aac`, `.ogg`, `.m4a`, `.wma`, `.aiff`, `.au`, `.ra`, `.3gp`, `.amr`, `.ac3`, `.dts`, `.ape`, `.mka`, `.opus`

### **1.2 📊 Annotations (CSV Files)**

Each audio file **must** have a corresponding CSV file with the **exact same base name**:

| Audio File | CSV File | Status |
|------------|----------|---------|
| `piano.wav` | `piano.csv` | ✅ Correct |
| `flute.mp3` | `flute_annotations.csv` | ⛔ Incorrect - name mismatch! |

The CSV annotation files must follow this specific structure:

- **Header row** with parameter names (must match those in `parameters.json`)
- **One row per frame** (75 frames per second)
- **Continuous parameters**: Numeric values (will be normalized to [0,1] range)
- **Categorical parameters**: Non-negative integers representing each class (0, 1, 2, ...)

Example CSV:

| saturation | reverb | instrument |
|------------|--------|------------|
| 10.5       | 40.3   | 0          |
| 10.0       | 40.32  | 1          |
| 9.8        | 40.25  | 0          |
| ...        | ...    | ...        |

### **1.3 ⚙️ Parameter Specifications**

Information about the parameters to be controlled must be stored in a `parameters.json` file. Each parameter must specify its **name** and **type** (either `continuous` or `class`). Additional features will depepnd on the type of the parameter as follow:

**🔢 Continuous Parameters** (numeric values like tempo, volume, saturation):
- `min`/`max`: Range used for normalization
- `unit`: Physical unit (e.g., bpm, %, dB)

**🏷️ Class Parameters** (categorical values like instrument type, genre):
- `classes`: Ordered list of class names (order determines integer mapping)

#### Example `parameters.json`:

```json
{
    "parameter_1": {"name": "saturation", "type": "continuous", "unit": "dB", "min": 0,   "max": 12}, 
    "parameter_2": {"name": "reverb",     "type": "continuous", "unit": "%",  "min": 0,   "max": 100},
    "parameter_3": {"name": "instrument", "type": "class",      "classes": ["piano", "flute"]}
}
```
## **📁 2. Required Data Structure**

Before running this notebook on your data, make sure it is structured as follows:

**Option 1:** Simple Structure (all data together)

```
dataset_folder/
└── raw/                         # Raw input data
    ├── parameters.json          # Parameter configuration file
    ├── piano.wav                # Audio files (various formats supported)
    ├── piano.csv                # Corresponding CSV annotations
    ├── flute.mp3                # More audio files...
    ├── flute.csv                # More CSV files...
    └── ...
```

**Option 2:** Split Structure (separate train/validation/test sets)
If you want to use specific data splits for training, validation, and testing.

```
dataset_folder/
└── raw/                         # Raw input data
    ├── parameters.json          # Parameter configuration file
    ├── train/
    │   ├── piano.wav            # Training audio files
    │   ├── piano.csv            # Training CSV annotations
    │   └── ...
    ├── validation/
    │   ├── flute.mp3            # Validation audio files
    │   ├── flute.csv            # Validation CSV files
    │   └── ...
    └── test/
        ├── violin.mp3           # Test audio files
        ├── violin.csv           # Test CSV files
        └── ...
```

## **🔄 3. Dataset Creation Pipeline**

Once your data is properly arranged, follow this notebook to create your dataset. The full pipeline consists of five steps:

- **📊 Visualization** - Analyze your raw data structure and parameters to verify everything is in order
- **🔧 Normalization** - Standardize format and normalize your data
- **🎵 EnCodec Encoding** - Convert audio to a compressed latent format that RNeNcodec can process
- **📄 Sidecar Creation**- Align parameters with audio frames (75 fps)
- **🤗 HuggingFace Dataset** - Package everything into a training-ready format

Let's get started! 👇

In [1]:
# Make repo root importable for this session (zero packaging)
import sys, pathlib
repo_root = pathlib.Path.cwd().parent if (pathlib.Path.cwd().name == "quickstart") else pathlib.Path.cwd()
sys.path.insert(0, str(repo_root))
%load_ext autoreload
%autoreload 2

dataset_path = "../data/trumpet1/" # Path to your dataset directory

### **3.1 📊 Visualization**

Visualize your data and make sure everything is alright.

In [ ]:
# Import visualization functions
from dataprep.step_0_visualization import quick_analyze

# 🔍 ANALYZE ENTIRE DATASET
quick_analyze(dataset_path, individual_plot_selector=True) # Set individual_plot_selector=True to choose which files to plot.

DATASET SUMMARY:

✅ Found 49 audio-CSV pairs
📁 Parameters: pitch
⏱️ Total audio duration: 1176.0 seconds

FILE DETAILS:

✅ 448-2: 24.0s, 1800 frames
✅ 302-2: 24.0s, 1800 frames
✅ 300-1: 24.0s, 1800 frames
✅ 267-3: 24.0s, 1800 frames
✅ 280-1: 24.0s, 1800 frames
✅ 279-4: 24.0s, 1800 frames
✅ 282-2: 24.0s, 1800 frames
✅ 299-3: 24.0s, 1800 frames
✅ 280-3: 24.0s, 1800 frames
✅ 255-2: 24.0s, 1800 frames
✅ 375-2: 24.0s, 1800 frames
✅ 375-3: 24.0s, 1800 frames
✅ 257-1: 24.0s, 1800 frames
✅ 420-3: 24.0s, 1800 frames
✅ 236-2: 24.0s, 1800 frames
✅ 334-5: 24.0s, 1800 frames
✅ 257-3: 24.0s, 1800 frames
✅ 395-4: 24.0s, 1800 frames
✅ 268-1: 24.0s, 1800 frames
✅ 234-1: 24.0s, 1800 frames
✅ 236-3: 24.0s, 1800 frames
✅ 255-4: 24.0s, 1800 frames
✅ 420-4: 24.0s, 1800 frames
✅ 375-4: 24.0s, 1800 frames
✅ 334-1: 24.0s, 1800 frames
✅ 355-2: 24.0s, 1800 frames
✅ 316-4: 24.0s, 1800 frames
✅ 395-2: 24.0s, 1800 frames
✅ 420-6: 24.0s, 1800 frames
✅ 317-2: 24.0s, 1800 frames
✅ 418-5: 24.0s, 1800 frames
✅ 317-3: 24

### **3.2 🔧 Normalization**

This step prepares your audio files by:
1. **Resampling** all audio to 24kHz mono (required by EnCodec)
2. **RMS Normalization** (optional) to ensure consistent loudness across your dataset (you may want apply_rms_normalization=False if the relative volume between your dataset files is important).

In [3]:
from dataprep.step_1_normalization import quick_normalize

quick_normalize(dataset_path, apply_rms_normalization=True)


DATA SUMMARY:

Found 49 audio-CSV pairs
Target peak windowed RMS: 0.1
Window size: 250ms
Input: ../data/trumpet1/raw
Output: ../data/trumpet1/normalized

DATA PROCESSING:

Processing: 234-1.wav
    Current peak RMS: 0.183799
    Target peak RMS: 0.100000
    Applying gain: -5.29 dB
    ✓ Saved: 234-1.wav
    ✅ Copied CSV: 234-1.csv → ../data/trumpet1/normalized/234-1.csv

Processing: 236-2.wav
    Current peak RMS: 0.185531
    Target peak RMS: 0.100000
    Applying gain: -5.37 dB
    ✓ Saved: 236-2.wav
    ✅ Copied CSV: 236-2.csv → ../data/trumpet1/normalized/236-2.csv

Processing: 236-3.wav
    Current peak RMS: 0.172115
    Target peak RMS: 0.100000
    Applying gain: -4.72 dB
    ✓ Saved: 236-3.wav
    ✅ Copied CSV: 236-3.csv → ../data/trumpet1/normalized/236-3.csv

Processing: 237-4.wav
    Current peak RMS: 0.246539
    Target peak RMS: 0.100000
    Applying gain: -7.84 dB
    ✓ Saved: 237-4.wav
    ✅ Copied CSV: 237-4.csv → ../data/trumpet1/normalized/237-4.csv

Processing: 255

### **3.3 🎵 EnCodec Encoding**

In [4]:
from dataprep.step_2_encodec import quick_encode

quick_encode(dataset_path, bandwidth=6.0, device="cpu")  # Force 8 codebooks and CPU


DATA PROCESSING:

Found 49 WAV files under ../data/trumpet1/normalized
Using device: cpu
Loading EnCodec model (facebook/encodec_24khz)...


Loading weights:   0%|          | 0/252 [00:00<?, ?it/s]

The image processor of type `EncodecImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
`use_fast` is set to `True` but the image processor class does not have a fast version.  Falling back to the slow version.
Encoding: 100%|██████████| 49/49 [00:00<00:00, 8974.32file/s]


DATA SUMMARY:

📁 234-1.ecdc
   Audio codes shape: (1, 1, 8, 1800)
   Frames: unknown
   Codebooks: unknown
   Original length: 575988 samples

📁 236-2.ecdc
   Audio codes shape: (1, 1, 8, 1800)
   Frames: unknown
   Codebooks: unknown
   Original length: 575988 samples

📁 236-3.ecdc
   Audio codes shape: (1, 1, 8, 1800)
   Frames: unknown
   Codebooks: unknown
   Original length: 575988 samples

📁 237-4.ecdc
   Audio codes shape: (1, 1, 8, 1800)
   Frames: unknown
   Codebooks: unknown
   Original length: 575988 samples

📁 255-2.ecdc
   Audio codes shape: (1, 1, 8, 1800)
   Frames: unknown
   Codebooks: unknown
   Original length: 575988 samples

📁 255-4.ecdc
   Audio codes shape: (1, 1, 8, 1800)
   Frames: unknown
   Codebooks: unknown
   Original length: 575988 samples

📁 257-1.ecdc
   Audio codes shape: (1, 1, 8, 1800)
   Frames: unknown
   Codebooks: unknown
   Original length: 575988 samples

📁 257-3.ecdc
   Audio codes shape: (1, 1, 8, 1800)
   Frames: unknown
   Codebooks: unkn

### **3.4 📄 Sidecar Creation**

In [6]:
from dataprep.step_3_sidecars import quick_create_sidecars

results = quick_create_sidecars(dataset_path)


SIDECAR CREATION:

📁 Tokens directory: ../data/trumpet1/tokens
📁 Raw directory: ../data/trumpet1/raw
📊 Found 49 .ecdc-CSV pairs
🎛️ Parameters: pitch

SIDECARS SUMMARY:

📄 355-2 sidecar:
    EnCodec frames: 1800
    CSV annotations: 1800
    ✅ Perfect alignment!
    ✅ Sidecar created (1800 frames, 1 features)

📄 255-2 sidecar:
    EnCodec frames: 1800
    CSV annotations: 1800
    ✅ Perfect alignment!
    ✅ Sidecar created (1800 frames, 1 features)

📄 236-2 sidecar:
    EnCodec frames: 1800
    CSV annotations: 1800
    ✅ Perfect alignment!
    ✅ Sidecar created (1800 frames, 1 features)

📄 257-3 sidecar:
    EnCodec frames: 1800
    CSV annotations: 1800
    ✅ Perfect alignment!
    ✅ Sidecar created (1800 frames, 1 features)

📄 396-4 sidecar:
    EnCodec frames: 1800
    CSV annotations: 1800
    ✅ Perfect alignment!
    ✅ Sidecar created (1800 frames, 1 features)

📄 280-3 sidecar:
    EnCodec frames: 1800
    CSV annotations: 1800
    ✅ Perfect alignment!
    ✅ Sidecar created (1800

### **3.5 🤗 HuggingFace Dataset**
(on Windows, you can ignore the "Failed simlink" messages)

In [7]:
from dataprep.step_4_HF import quick_create_dataset

results = quick_create_dataset(dataset_path)


HUGGINGFACE DATASET CREATION:

📁 Tokens directory: ../data/trumpet1/tokens
📁 Output directory: ../data/trumpet1/hf_dataset
🔗 Materialize mode: link
🧹 Cleaning up dataset directory: ../data/trumpet1/hf_dataset/tokens
✅ Cleaned up tokens directory

📊 No split structure detected
📌 Using all data as 'train' split

   📂 train: 49 .ecdc files found

📋 Saved expanded conditioning config to: conditioning_config.json
   • Features: pitch
   • Total features: 1

💾 Saved DatasetDict to: ../data/trumpet1/hf_dataset
📈 Dataset summary:
   • train: 49 samples

🔍 Verifying dataset files...
✅ All audio files verified successfully
